# RPM Control PID Analysis
Analyzes RpmControlLogger CSV data and recommends P, I, D values

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set style for better plots
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 10)

## Load and Parse CSV Data

In [ ]:
# Find the most recent RPM log file
log_dir = Path('../TeamCode/src/main/java/org/firstinspires/ftc/teamcode').parent.parent.parent / 'rpm_logs'
# Or specify manually: log_file = 'path/to/your/log/file.csv'

# For now, ask user to specify the file
log_file = 'rpm_log.csv'  # Change this to your actual log file path

try:
    df = pd.read_csv(log_file)
    print(f"✓ Loaded {len(df)} samples from {log_file}")
    print(f"\nColumns: {list(df.columns)}")
    print(f"\nFirst few rows:")
    print(df.head())
except FileNotFoundError:
    print(f"⚠ File not found: {log_file}")
    print("Please place your RPM log CSV file here or update the log_file path.")

## Data Analysis

In [ ]:
# Basic statistics
print("="*50)
print("RPM CONTROL STATISTICS")
print("="*50)

print(f"\nTest Duration: {df['time_s'].max():.2f} seconds")
print(f"Target RPM: {df['targetRPM'].iloc[0]:.1f}")
print(f"\nActual RPM1:")
print(f"  Mean: {df['actualRPM1'].mean():.1f}")
print(f"  Max:  {df['actualRPM1'].max():.1f}")
print(f"  Min:  {df['actualRPM1'].min():.1f}")

print(f"\nActual RPM2:")
print(f"  Mean: {df['actualRPM2'].mean():.1f}")
print(f"  Max:  {df['actualRPM2'].max():.1f}")
print(f"  Min:  {df['actualRPM2'].min():.1f}")

# Average actual RPM
df['actualRPM_avg'] = (df['actualRPM1'] + df['actualRPM2']) / 2

print(f"\nError Statistics:")
print(f"  Mean Error: {df['error'].mean():.1f} RPM")
print(f"  Std Dev:    {df['error'].std():.1f} RPM")
print(f"  Max Error:  {df['error'].abs().max():.1f} RPM")

# Settling time (error within 5% of target)
target_rpm = df['targetRPM'].iloc[0]
settling_threshold = abs(target_rpm) * 0.05  # 5% of target
settled_idx = (df['error'].abs() <= settling_threshold).idxmax()
if settled_idx > 0:
    settling_time = df['time_s'].iloc[settled_idx]
    print(f"\nSettling Time (5% error): {settling_time:.2f}s")
else:
    print(f"\nSettling Time: Did not settle to 5%")

print(f"\nPower Statistics:")
print(f"  Mean Power: {df['power'].mean():.3f}")
print(f"  Max Power:  {df['power'].max():.3f}")
print(f"  Min Power:  {df['power'].min():.3f}")

## Visualization

In [ ]:
# Create comprehensive plots
fig, axes = plt.subplots(3, 2, figsize=(16, 12))

# Plot 1: Target vs Actual RPM
ax = axes[0, 0]
ax.plot(df['time_s'], df['targetRPM'], 'r--', linewidth=2, label='Target', alpha=0.7)
ax.plot(df['time_s'], df['actualRPM1'], 'b-', linewidth=1.5, label='Motor 1', alpha=0.8)
ax.plot(df['time_s'], df['actualRPM2'], 'g-', linewidth=1.5, label='Motor 2', alpha=0.8)
ax.plot(df['time_s'], df['actualRPM_avg'], 'purple', linewidth=2, label='Average', alpha=0.8)
ax.set_xlabel('Time (s)')
ax.set_ylabel('RPM')
ax.set_title('Target vs Actual RPM')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Error over time
ax = axes[0, 1]
ax.plot(df['time_s'], df['error'], 'r-', linewidth=2)
ax.axhline(y=0, color='k', linestyle='--', alpha=0.3)
ax.fill_between(df['time_s'], df['error'], alpha=0.3)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Error (RPM)')
ax.set_title('RPM Error Over Time')
ax.grid(True, alpha=0.3)

# Plot 3: Power output
ax = axes[1, 0]
ax.plot(df['time_s'], df['power'], 'b-', linewidth=2)
ax.axhline(y=1.0, color='r', linestyle='--', alpha=0.3, label='Max')
ax.axhline(y=-1.0, color='r', linestyle='--', alpha=0.3, label='Min')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Throttle')
ax.set_title('Power Output')
ax.set_ylim([-1.2, 1.2])
ax.grid(True, alpha=0.3)

# Plot 4: PID Output vs Power
ax = axes[1, 1]
ax.plot(df['time_s'], df['pidOutput'], 'purple', linewidth=2, label='PID Output (raw)', alpha=0.8)
ax.plot(df['time_s'], df['power'], 'b-', linewidth=2, label='Power (clamped)', alpha=0.8)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Value')
ax.set_title('PID Output vs Final Power')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 5: Integral Error accumulation
ax = axes[2, 0]
ax.plot(df['time_s'], df['integralError'], 'g-', linewidth=2)
ax.axhline(y=0, color='k', linestyle='--', alpha=0.3)
ax.fill_between(df['time_s'], df['integralError'], alpha=0.3, color='green')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Integral Error')
ax.set_title('Accumulated Integral Error (I term)')
ax.grid(True, alpha=0.3)

# Plot 6: Error Phase Space (Error vs dError/dt)
ax = axes[2, 1]
# Calculate derivative of error
derror = df['error'].diff() / df['time_s'].diff()
scatter = ax.scatter(df['error'], derror, c=df['time_s'], cmap='viridis', s=30, alpha=0.6)
ax.axhline(y=0, color='k', linestyle='--', alpha=0.3)
ax.axvline(x=0, color='k', linestyle='--', alpha=0.3)
ax.set_xlabel('Error (RPM)')
ax.set_ylabel('dError/dt (RPM/s)')
ax.set_title('Phase Space Plot (Error vs Error Rate)')
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Time (s)')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('rpm_control_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved plot to rpm_control_analysis.png")

## PID Tuning Analysis & Recommendations

In [ ]:
print("\n" + "="*60)
print("PID TUNING ANALYSIS & RECOMMENDATIONS")
print("="*60)

# Analyze system response characteristics
settling_threshold = abs(target_rpm) * 0.05
rise_time_threshold = abs(target_rpm) * 0.90

# Rise time (time to reach 90% of target)
rise_idx = (df['actualRPM_avg'].abs() >= rise_time_threshold).idxmax()
rise_time = df['time_s'].iloc[rise_idx] if rise_idx > 0 else np.inf

# Overshoot detection
max_rpm = df['actualRPM_avg'].max() if target_rpm > 0 else df['actualRPM_avg'].min()
overshoot = ((max_rpm - abs(target_rpm)) / abs(target_rpm) * 100) if target_rpm != 0 else 0

# Error characteristics
mean_error = df['error'].mean()
error_std = df['error'].std()
max_error = df['error'].abs().max()

print(f"\n📊 SYSTEM RESPONSE CHARACTERISTICS:")
print(f"   Rise Time (90%):        {rise_time:.2f}s")
print(f"   Settling Time (5%):     {settling_time:.2f}s" if 'settling_time' in locals() else "   Settling Time:         Did not settle")
print(f"   Overshoot:              {overshoot:.1f}%")
print(f"   Mean Error:             {mean_error:.1f} RPM")
print(f"   Max Error:              {max_error:.1f} RPM")

# Diagnose current behavior
print(f"\n🔍 DIAGNOSIS:")

if overshoot > 10:
    print(f"   ⚠ High overshoot ({overshoot:.1f}%) - P or I too aggressive")
elif overshoot < -10:
    print(f"   ⚠ High undershoot - P too weak")
else:
    print(f"   ✓ Overshoot within acceptable range ({overshoot:.1f}%)")

if error_std > abs(target_rpm) * 0.1:
    print(f"   ⚠ High oscillation (σ={error_std:.1f}) - D may help")
else:
    print(f"   ✓ Low oscillation")

if abs(mean_error) > abs(target_rpm) * 0.05:
    print(f"   ⚠ Steady-state error ({mean_error:.1f}) - Need more I")
else:
    print(f"   ✓ Good steady-state accuracy")

# Calculate recommended PID values
print(f"\n💡 CURRENT PID VALUES:")
print(f"   P = 0.001")
print(f"   I = 0.0002")
print(f"   D = 0.0")

print(f"\n📈 RECOMMENDED PID ADJUSTMENTS:")

# Base recommendations
kp_current = 0.001
ki_current = 0.0002
kd_current = 0.0

kp_rec = kp_current
ki_rec = ki_current
kd_rec = kd_current

# Adjust based on observations
if rise_time > 2.0 or max_error > abs(target_rpm) * 0.3:
    print(f"   • Increase P: Response is too slow")
    kp_rec = kp_current * 1.5
elif overshoot > 15 or error_std > abs(target_rpm) * 0.15:
    print(f"   • Decrease P: System is oscillating too much")
    kp_rec = kp_current * 0.8
else:
    print(f"   • P is good: {kp_current}")

if abs(mean_error) > abs(target_rpm) * 0.05:
    print(f"   • Increase I: Reduce steady-state error")
    ki_rec = ki_current * 2.0
elif abs(mean_error) < abs(target_rpm) * 0.01 and error_std < abs(target_rpm) * 0.05:
    print(f"   • I is good: {ki_current}")
else:
    print(f"   • Fine-tune I: {ki_current} → {ki_current * 1.2}")
    ki_rec = ki_current * 1.2

if error_std > abs(target_rpm) * 0.1 and overshoot > 5:
    print(f"   • Add D: Dampen oscillations")
    kd_rec = 0.001
else:
    print(f"   • D can stay at 0 (or increase if more damping needed)")

print(f"\n🎯 RECOMMENDED PID VALUES:")
print(f"   P = {kp_rec:.6f} (current: {kp_current})")
print(f"   I = {ki_rec:.6f} (current: {ki_current})")
print(f"   D = {kd_rec:.6f} (current: {kd_current})")

print(f"\n💾 UPDATE YOUR CODE:")
print(f"   private static final double KP = {kp_rec:.6f};")
print(f"   private static final double KI = {ki_rec:.6f};")
print(f"   private static final double KD = {kd_rec:.6f};")

print(f"\n📝 TUNING TIPS:")
print(f"   1. Adjust P first: Start with 1.5x or 0.8x")
print(f"   2. If steady-state error persists: Increase I")
print(f"   3. If oscillating too much: Add D or reduce P/I")
print(f"   4. Test in small increments (±10-20% at a time)")
print(f"   5. Collect multiple logs to verify consistency")

## Export Summary

In [ ]:
# Save analysis summary to text file
summary_file = 'pid_tuning_summary.txt'

with open(summary_file, 'w') as f:
    f.write("RPM CONTROL PID TUNING SUMMARY\n")
    f.write("="*50 + "\n\n")
    f.write(f"Log File: {log_file}\n")
    f.write(f"Test Duration: {df['time_s'].max():.2f}s\n")
    f.write(f"Target RPM: {target_rpm:.1f}\n\n")
    f.write(f"RECOMMENDED PID VALUES:\n")
    f.write(f"P = {kp_rec:.6f}\n")
    f.write(f"I = {ki_rec:.6f}\n")
    f.write(f"D = {kd_rec:.6f}\n")

print(f"✓ Saved summary to {summary_file}")